<a href="https://colab.research.google.com/github/MaSBroEarl/-.-Text-classification/blob/main/Classic_NLP_%D0%BF%D0%BE%D0%B4%D1%85%D0%BE%D0%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импортируем библиотеки

In [ ]:
# Импорт
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Загружаем данные

In [ ]:
# Данные
df = pd.read_csv('/content/train.csv')

# **EDA**

In [ ]:
print("\n1. РАЗМЕР ДАННЫХ")
print("-"*40)
print(f"Строк: {df.shape[0]}")
print(f"Колонок: {df.shape[1]}")
print(f"Колонки: {df.columns.tolist()}")

print("\n2. ТИПЫ ДАННЫХ")
print("-"*40)
print(df.dtypes)


1. РАЗМЕР ДАННЫХ
----------------------------------------
Строк: 7613
Колонок: 5
Колонки: ['id', 'keyword', 'location', 'text', 'target']

2. ТИПЫ ДАННЫХ
----------------------------------------
id           int64
keyword     object
location    object
text        object
target       int64
dtype: object


In [ ]:
# Смотрим пропуски
print("\n3. ПРОПУСКИ В ДАННЫХ")
print("-"*40)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
for col in df.columns:
    print(f"{col:10} -> {missing[col]:4} пропусков ({missing_pct[col]:5.1f}%)")


3. ПРОПУСКИ В ДАННЫХ
----------------------------------------
id         ->    0 пропусков (  0.0%)
keyword    ->   61 пропусков (  0.8%)
location   -> 2533 пропусков ( 33.3%)
text       ->    0 пропусков (  0.0%)
target     ->    0 пропусков (  0.0%)


In [ ]:
print("\n4. СТАТИСТИКА ПО ТЕКСТАМ")
print("-"*40)

text_lengths = df['text'].str.len()
word_counts = df['text'].str.split().str.len()
hashtag_counts = df['text'].str.count('#')
mention_counts = df['text'].str.count('@')
url_counts = df['text'].str.count('http')

print(f"Длина текста (символы):")
print(f"  Средняя: {text_lengths.mean():.1f}")
print(f"  Мин: {text_lengths.min()}")
print(f"  Макс: {text_lengths.max()}")

print(f"\nКоличество слов:")
print(f"  Среднее: {word_counts.mean():.1f}")
print(f"  Мин: {word_counts.min()}")
print(f"  Макс: {word_counts.max()}")

print(f"\nХэштеги (#):")
print(f"  Среднее: {hashtag_counts.mean():.2f}")
print(f"  Макс: {hashtag_counts.max()}")

print(f"\nУпоминания (@):")
print(f"  Среднее: {mention_counts.mean():.2f}")
print(f"  Макс: {mention_counts.max()}")

print(f"\nСсылки (http):")
print(f"  Среднее: {url_counts.mean():.2f}")
print(f"  Макс: {url_counts.max()}")


4. СТАТИСТИКА ПО ТЕКСТАМ
----------------------------------------
Длина текста (символы):
  Средняя: 101.0
  Мин: 7
  Макс: 157

Количество слов:
  Среднее: 14.9
  Мин: 1
  Макс: 31

Хэштеги (#):
  Среднее: 0.45
  Макс: 13

Упоминания (@):
  Среднее: 0.36
  Макс: 8

Ссылки (http):
  Среднее: 0.62
  Макс: 4


In [ ]:
# Анализ целевой переменной
print("\n5. АНАЛИЗ TARGET")
print("-"*40)

target_counts = df['target'].value_counts()
print(f"Распределение классов:")
for val, count in target_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {val}: {count} ({pct:.1f}%)")

print(f"\nДоля класса 1: {df['target'].mean()*100:.1f}%")
print(f"Доля класса 0: {(1 - df['target'].mean())*100:.1f}%")

# Проверка на дисбаланс
if df['target'].nunique() == 1:
    print("\n⚠️ ВНИМАНИЕ: ТОЛЬКО ОДИН КЛАСС (все target = 1)!")
    print("   Нет отрицательных примеров для обучения.")


5. АНАЛИЗ TARGET
----------------------------------------
Распределение классов:
  0: 4342 (57.0%)
  1: 3271 (43.0%)

Доля класса 1: 43.0%
Доля класса 0: 57.0%


In [ ]:
print("\n6. ДУБЛИКАТЫ")
print("-"*40)
print(f"Дубликатов по id: {df['id'].duplicated().sum()}")
print(f"Дубликатов по тексту: {df['text'].duplicated().sum()}")


6. ДУБЛИКАТЫ
----------------------------------------
Дубликатов по id: 0
Дубликатов по тексту: 110


In [ ]:
def quick_eda(df):
    """Быстрый EDA"""
    print("="*60)
    print("БЫСТРЫЙ EDA")
    print("="*60)

    print(f"\nРазмер: {df.shape}")
    print(f"\nПропуски:\n{df.isnull().sum()}")

    print(f"\nДлина текстов:")
    print(f"  Средняя: {df['text'].str.len().mean():.1f}")
    print(f"  Мин: {df['text'].str.len().min()}")
    print(f"  Макс: {df['text'].str.len().max()}")

    print(f"\nTarget:")
    print(df['target'].value_counts())
    print(f"  Доля 1: {df['target'].mean():.2%}")

    # Топ-слова
    all_text = ' '.join(df['text'].str.lower())
    words = re.findall(r'\b\w{3,}\b', all_text)
    stop_words = {'the','and','for','are','but','not','you','all','can','had','her','was','one','our','out'}
    words = [w for w in words if w not in stop_words]
    print(f"\nТоп-5 слов: {Counter(words).most_common(5)}")

    print("\n" + "="*60)

# Использование:
quick_eda(df)

БЫСТРЫЙ EDA

Размер: (7613, 5)

Пропуски:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Длина текстов:
  Средняя: 101.0
  Мин: 7
  Макс: 157

Target:
target
0    4342
1    3271
Name: count, dtype: int64
  Доля 1: 42.97%

Топ-5 слов: [('http', 4309), ('that', 623), ('with', 572), ('this', 480), ('from', 422)]



Выводы по EDA:
Данные сбалансированы: класс 0 (не ЧП) — 57%, класс 1 (ЧП) — 43%. Дисбаланс небольшой, можно работать без специальных техник.

Проблемы с качеством:

    33% пропусков в location (2533 записи)

    0.8% пропусков в keyword (61 запись)

    110 дубликатов текстов (1.4% данных)

Тексты короткие: в среднем 101 символ, 15 слов — идеально для быстрых моделей (TF-IDF + Logistic Regression)

Технические особенности:

    Много ссылок (в среднем 0.62 на твит) — их нужно чистить

    Хэштеги и упоминания есть, но не доминируют

Странность в топ-словах: "http" на первом месте (4309 раз) — это артефакт, нужно удалять

# 1. Classic NLP подход


Препроцессинг, токенизация, фильтрация

In [ ]:
 # Необходимо дропнуть колонки
# Делаем дроп id, location(33% пропусков)
# text, keyword и target оставляем
df = df.drop(['id', 'location'], axis=1)
df['keyword'] = df['keyword'].fillna('')

In [ ]:
stop_words = set(stopwords.words('english')) # Загружаем стоп-слова

# ДОБАВЛЯЕМ СПЕЦИФИЧНЫЕ СТОП-СЛОВА ДЛЯ ТВИТОВ
extra_stop_words = {
    'like', 'amp', 'get', 'via', 'got', 'just', 'really', 'even', 'well',
    'now', 'still', 'one', 'would', 'back', 'time', 'new', 'day', 'today',
    'also', 'yet', 'however', 'though', 'much', 'more', 'many', 'lot',
    'im', 'ive', 'ill', 'youre', 'theyre', 'were', 'could', 'should',
    'http', 'https', 'www', 'com', 'rt', 'good', 'great', 'big', 'first',
    'last', 'next', 'another', 'any', 'every', 'some', 'something',
    'said', 'says', 'going', 'want', 'need', 'know', 'think', 'see', 'look'
}
stop_words.update(extra_stop_words)

print(f"Всего стоп-слов: {len(stop_words)}")
print(f"Примеры: {list(stop_words)[:10]}")


Всего стоп-слов: 250
Примеры: ['big', 'did', 'really', 'shan', 'via', 'ma', "haven't", 've', 'www', 'because']


In [ ]:
def clean_text_with_stopwords(text):
    """
    Очистка текста с удалением стоп-слов
    """
    # Приводим к нижнему регистру
    text = str(text).lower()

    # Удаляем ссылки
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Удаляем упоминания (@username)
    text = re.sub(r'@\w+', '', text)

    # Удаляем хэштеги (оставляем слова)
    text = re.sub(r'#', '', text)

    # Удаляем пунктуацию
    text = re.sub(r'[^\w\s]', '', text)

    # Удаляем цифры
    text = re.sub(r'\d+', '', text)

    # Разбиваем на слова
    words = text.split()

    # Удаляем стоп-слова и короткие слова (<=2 символа)
    words = [word for word in words if word not in stop_words and len(word) > 2]

    # Собираем обратно
    return ' '.join(words)

# Применяем очистку
df['clean_text'] = df['text'].apply(clean_text_with_stopwords)

In [ ]:
#  ОБЪЕДИНЯЕМ С KEYWORD
df['text_for_model'] = df['keyword'] + ' ' + df['clean_text']
df['text_for_model'] = df['text_for_model'].str.strip()

In [ ]:
# Удаляем дубликаты
duplicates_before = df.duplicated(subset=['text']).sum()
df = df.drop_duplicates(subset=['text'])
print(f"Удалено дубликатов: {duplicates_before}")

Удалено дубликатов: 110


In [ ]:
all_words = []
for text in df['clean_text']:
    all_words.extend(text.split())

word_freq = Counter(all_words)

print("\n" + "="*60)
print("ТОП-20 СЛОВ ПОСЛЕ УДАЛЕНИЯ СТОП-СЛОВ")
print("="*60)
for i, (word, count) in enumerate(word_freq.most_common(20), 1):
    print(f"{i:2}. {word:15} -> {count:5}")


ТОП-20 СЛОВ ПОСЛЕ УДАЛЕНИЯ СТОП-СЛОВ
 1. fire            ->   245
 2. news            ->   193
 3. dont            ->   191
 4. people          ->   190
 5. video           ->   161
 6. emergency       ->   156
 7. disaster        ->   147
 8. police          ->   138
 9. body            ->   124
10. crash           ->   118
11. storm           ->   117
12. california      ->   116
13. man             ->   110
14. burning         ->   110
15. suicide         ->   110
16. world           ->   103
17. cant            ->   102
18. nuclear         ->   101
19. love            ->   100
20. fires           ->   100


Лемматизация

In [ ]:
# Инициализируем лемматизатор
lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(text):
    """
    Токенизация + лемматизация + удаление стоп-слов
    """
    # Токенизация
    tokens = word_tokenize(text.lower())

    # Фильтрация: только буквы, не стоп-слова, длина > 2
    tokens = [token for token in tokens
              if token.isalpha()
              and token not in stop_words
              and len(token) > 2]

    # Лемматизация
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return tokens

Токенизация

In [ ]:
# Применяем токенизацию
df['tokens'] = df['text_for_model'].apply(tokenize_and_lemmatize)

# Собираем обратно в текст (для векторизации)
df['text_tokenized'] = df['tokens'].apply(lambda x: ' '.join(x))

# Проверяем результат
print(f"Пример текста: {df['text_for_model'].iloc[0]}")
print(f"Токены: {df['tokens'].iloc[0]}")
print(f"После лемматизации: {df['text_tokenized'].iloc[0]}")

# Статистика по токенам
all_tokens = []
for tokens in df['tokens']:
    all_tokens.extend(tokens)

print(f"\nВсего токенов: {len(all_tokens)}")
print(f"Уникальных токенов: {len(set(all_tokens))}")
print(f"Среднее количество токенов: {df['tokens'].apply(len).mean():.1f}")

# Топ-токенов после лемматизации
print("\nТоп-10 токенов после лемматизации:")
for word, count in Counter(all_tokens).most_common(10):
    print(f"  {word}: {count}")

Пример текста: deeds reason earthquake may allah forgive
Токены: ['deed', 'reason', 'earthquake', 'may', 'allah', 'forgive']
После лемматизации: deed reason earthquake may allah forgive

Всего токенов: 65968
Уникальных токенов: 13093
Среднее количество токенов: 8.8

Топ-10 токенов после лемматизации:
  fire: 415
  emergency: 262
  body: 254
  suicide: 204
  building: 196
  people: 193
  news: 193
  dont: 191
  disaster: 186
  death: 185


Делим данные на train и test

In [ ]:
X = df['text_for_model']
y = df['target']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Train: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")


Train: 6002 (80.0%)
Test:  1501 (20.0%)


Векторизация

In [ ]:
#TF-IDF Векторизация
vectorizer = TfidfVectorizer(
    max_features=5000,          # Топ-5000 слов
    ngram_range=(1, 2),         # Униграммы + биграммы
    min_df=3,                   # Игнорируем слова, встречающиеся < 3 раз
    max_df=0.8,                 # Игнорируем слова, встречающиеся > 80% текстов
    stop_words='english',       # Встроенные стоп-слова (дополнительно)
    sublinear_tf=True           # Используем 1+log(tf)
)

In [ ]:
# Обучаем на тренировочных данных
X_train_tfidf = vectorizer.fit_transform(X_train)
print(f"Train TF-IDF shape: {X_train_tfidf.shape}")

Train TF-IDF shape: (6002, 4877)


In [ ]:
# Transform на тестовых данных
X_test_tfidf = vectorizer.transform(X_test)
print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

Test TF-IDF shape: (1501, 4877)


In [ ]:
# Получаем названия признаков
feature_names = vectorizer.get_feature_names_out()
print(f"\nКоличество признаков: {len(feature_names)}")


Количество признаков: 4877


In [ ]:
# Показываем топ-10 признаков
print("\nТоп-10 признаков (слов):")
for i, feature in enumerate(feature_names[:10]):
    print(f"  {i+1}. {feature}")


Топ-10 признаков (слов):
  1. 20accident
  2. 20accident airplane
  3. 20accident experts
  4. 20accident horrible
  5. 20bag
  6. 20bag ladies
  7. 20bag louis
  8. 20bagging
  9. 20bagging body
  10. 20bagging drake


In [ ]:
# Проверка разреженности
# Показываем топ-10 признаков
# Плотность матрицы (сколько ненулевых элементов)
density_train = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
density_test = X_test_tfidf.nnz / (X_test_tfidf.shape[0] * X_test_tfidf.shape[1])

print(f"Плотность train: {density_train:.4f} ({density_train*100:.2f}%)")
print(f"Плотность test:  {density_test:.4f} ({density_test*100:.2f}%)")

# Среднее количество ненулевых элементов на документ
mean_nnz_train = X_train_tfidf.nnz / X_train_tfidf.shape[0]
mean_nnz_test = X_test_tfidf.nnz / X_test_tfidf.shape[0]

print(f"Среднее ненулевых признаков на документ (train): {mean_nnz_train:.1f}")
print(f"Среднее ненулевых признаков на документ (test):  {mean_nnz_test:.1f}")



Плотность train: 0.0015 (0.15%)
Плотность test:  0.0014 (0.14%)
Среднее ненулевых признаков на документ (train): 7.4
Среднее ненулевых признаков на документ (test):  6.9


ML модель

In [ ]:
#Обучаем логистическую регрессию
model = LogisticRegression(
    max_iter=1000,          # Максимум итераций для сходимости
    random_state=42,         # Для воспроизводимости
    C=1.0,                   # Обратная сила регуляризации (1.0 - стандарт)
    class_weight='balanced', # Учитываем дисбаланс классов
    solver='lbfgs',          # Оптимальный солвер для небольших данных
    penalty='l2'             # L2 регуляризация
)

In [ ]:
###
# Обучаем модель
model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [ ]:
# Предсказания на train (для проверки переобучения)
y_train_pred = model.predict(X_train_tfidf)
y_train_proba = model.predict_proba(X_train_tfidf)[:, 1]

In [ ]:
# Предсказания на test (главная оценка)
y_test_pred = model.predict(X_test_tfidf)
y_test_proba = model.predict_proba(X_test_tfidf)[:, 1]

оценка модели

In [ ]:
# Метрики для train
train_acc = accuracy_score(y_train, y_train_pred)
train_prec = precision_score(y_train, y_train_pred)
train_rec = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Метрики для test
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

print("📊 Метрики на TRAIN:")
print(f"  Accuracy:  {train_acc:.4f}")
print(f"  Precision: {train_prec:.4f}")
print(f"  Recall:    {train_rec:.4f}")
print(f"  F1-score:  {train_f1:.4f}")

print("\n📊 Метрики на TEST:")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"  F1-score:  {test_f1:.4f}")
print(f"  ROC-AUC:   {test_roc_auc:.4f}")

# Проверка на переобучение
overfit_gap = train_acc - test_acc
print(f"\n📊 Разница train-test: {overfit_gap:.4f}")
if overfit_gap > 0.05:
    print("⚠️ Есть небольшое переобучение")
else:
    print("✅ Переобучения нет")

📊 Метрики на TRAIN:
  Accuracy:  0.8615
  Precision: 0.8425
  Recall:    0.8303
  F1-score:  0.8364

📊 Метрики на TEST:
  Accuracy:  0.8048
  Precision: 0.7698
  Recall:    0.7734
  F1-score:  0.7716
  ROC-AUC:   0.8623

📊 Разница train-test: 0.0567
⚠️ Есть небольшое переобучение


In [ ]:
#  CLASSIFICATION REPORT
print("\n" + "="*60)
print("CLASSIFICATION REPORT (TEST)")
print("="*60)
print(classification_report(y_test, y_test_pred, target_names=['Не ЧП (0)', 'ЧП (1)']))


CLASSIFICATION REPORT (TEST)
              precision    recall  f1-score   support

   Не ЧП (0)       0.83      0.83      0.83       861
      ЧП (1)       0.77      0.77      0.77       640

    accuracy                           0.80      1501
   macro avg       0.80      0.80      0.80      1501
weighted avg       0.80      0.80      0.80      1501



Обучаем градиентный бустинг

In [ ]:
# Базовая модель XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,           # Количество деревьев
    max_depth=6,                # Глубина деревьев
    learning_rate=0.1,          # Скорость обучения
    subsample=0.8,              # Доля выборки для каждого дерева
    colsample_bytree=0.8,       # Доля признаков для каждого дерева
    random_state=42,
    use_label_encoder=False,    # Отключаем предупреждения
    eval_metric='logloss'       # Метрика для оценки
)

In [ ]:
# Обучаем модель
xgb_model.fit(
    X_train_tfidf,
    y_train,
    eval_set=[(X_train_tfidf, y_train), (X_test_tfidf, y_test)],
    verbose=False
)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:26:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
# Предсказания на train
y_train_pred_xgb = xgb_model.predict(X_train_tfidf)
y_train_proba_xgb = xgb_model.predict_proba(X_train_tfidf)[:, 1]

# Предсказания на test
y_test_pred_xgb = xgb_model.predict(X_test_tfidf)
y_test_proba_xgb = xgb_model.predict_proba(X_test_tfidf)[:, 1]

In [ ]:
# Train метрики
train_acc_xgb = accuracy_score(y_train, y_train_pred_xgb)
train_prec_xgb = precision_score(y_train, y_train_pred_xgb)
train_rec_xgb = recall_score(y_train, y_train_pred_xgb)
train_f1_xgb = f1_score(y_train, y_train_pred_xgb)

# Test метрики
test_acc_xgb = accuracy_score(y_test, y_test_pred_xgb)
test_prec_xgb = precision_score(y_test, y_test_pred_xgb)
test_rec_xgb = recall_score(y_test, y_test_pred_xgb)
test_f1_xgb = f1_score(y_test, y_test_pred_xgb)
test_roc_auc_xgb = roc_auc_score(y_test, y_test_proba_xgb)

print("📊 Метрики на TRAIN:")
print(f"  Accuracy:  {train_acc_xgb:.4f}")
print(f"  Precision: {train_prec_xgb:.4f}")
print(f"  Recall:    {train_rec_xgb:.4f}")
print(f"  F1-score:  {train_f1_xgb:.4f}")

print("\n📊 Метрики на TEST:")
print(f"  Accuracy:  {test_acc_xgb:.4f}")
print(f"  Precision: {test_prec_xgb:.4f}")
print(f"  Recall:    {test_rec_xgb:.4f}")
print(f"  F1-score:  {test_f1_xgb:.4f}")
print(f"  ROC-AUC:   {test_roc_auc_xgb:.4f}")

# Переобучение
overfit_gap_xgb = train_acc_xgb - test_acc_xgb
print(f"\n📊 Разница train-test: {overfit_gap_xgb:.4f}")
if overfit_gap_xgb > 0.05:
    print("⚠️ Есть небольшое переобучение")
else:
    print("✅ Переобучения нет")

📊 Метрики на TRAIN:
  Accuracy:  0.7854
  Precision: 0.9161
  Recall:    0.5465
  F1-score:  0.6846

📊 Метрики на TEST:
  Accuracy:  0.7548
  Precision: 0.8656
  Recall:    0.5031
  F1-score:  0.6364
  ROC-AUC:   0.8269

📊 Разница train-test: 0.0306
✅ Переобучения нет


Подсчет метрик

In [ ]:
print("\n" + "="*60)
print("🏆 ИТОГОВЫЙ ВЫБОР МОДЕЛИ")
print("="*60)

print("""
📊 СРАВНЕНИЕ ПО МЕТРИКАМ:

Метрика           | Logistic Regression | XGBoost
------------------|---------------------|---------
Accuracy          | 0.8048 ✅           | 0.7548
Precision         | 0.7698              | 0.8656 ✅
Recall            | 0.7734 ✅           | 0.5031
F1-score          | 0.7716 ✅           | 0.6364
ROC-AUC           | 0.8623 ✅           | 0.8269
Переобучение      | 5.67%               | 3.06% ✅
""")


🏆 ИТОГОВЫЙ ВЫБОР МОДЕЛИ

📊 СРАВНЕНИЕ ПО МЕТРИКАМ:

Метрика           | Logistic Regression | XGBoost 
------------------|---------------------|---------
Accuracy          | 0.8048 ✅           | 0.7548  
Precision         | 0.7698              | 0.8656 ✅
Recall            | 0.7734 ✅           | 0.5031  
F1-score          | 0.7716 ✅           | 0.6364  
ROC-AUC           | 0.8623 ✅           | 0.8269  
Переобучение      | 5.67%               | 3.06% ✅



Вывод: для данного датасета Logistic Regression показывает лучшие результаты и должна быть основной моделью. XGBoost не подходит из-за низкого Recall (пропускает слишком много ЧП)